# Reviewer 2 / matched-ablation training loop for Google Colab

This notebook follows the reviewer-required protocol for the final paper experiment rather than the earlier heavier smoke/full profile.

Main reporting configuration:
- 3 independent seeds: [1, 2, 3]
- 15 epochs per seed
- best QTDB validation checkpoint retained within each run
- fixed record-level splits preserved before any window generation
- matched A0-A4 ablation as the primary causal comparison

This is the computationally feasible protocol requested in the review notes. The earlier 5-seed, 30-epoch configuration is not the final reporting protocol and is not used for the matched ablation.


In [29]:
# Colab setup: install dependencies, mount Drive, and create persistent paths.
%pip -q install wfdb scipy scikit-learn pandas numpy torch

from pathlib import Path
import json, os, random, shutil, time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

try:
    from google.colab import drive
    IN_COLAB = True
except Exception:
    IN_COLAB = False


def first_existing(*paths):
    for path in paths:
        candidate = Path(path)
        if candidate.exists():
            return candidate
    return Path(paths[0])

if IN_COLAB:
    drive.mount('/content/drive')
    DRIVE_ROOT = Path('/content/drive/MyDrive/PQRST_Reviewer2')
    DATA_ROOT = DRIVE_ROOT / 'datasets'
    QTDB_DIR = DATA_ROOT / 'qtdb-1.0.0'
    LUDB_DIR = DATA_ROOT / 'ludb-1.0.1'
    ARTIFACT_DIR = DRIVE_ROOT / 'reviewer2_artifacts'
else:
    QTDB_DIR = first_existing(
        Path('/Users/vishwam/VSCode/physionet.org/files/qtdb/1.0.0'),
        Path('/Users/vishwam/VSCode/PQRST_mapping/physionet.org/files/qtdb/1.0.0'),
        Path.cwd() / 'physionet.org/files/qtdb/1.0.0',
        Path.cwd() / 'PQRST_mapping/physionet.org/files/qtdb/1.0.0',
    )
    LUDB_DIR = first_existing(
        Path('/Users/vishwam/VSCode/physionet.org/files/ludb/1.0.1/data'),
        Path('/Users/vishwam/VSCode/PQRST_mapping/physionet.org/files/ludb/1.0.1/data'),
        Path.cwd() / 'physionet.org/files/ludb/1.0.1/data',
        Path.cwd() / 'PQRST_mapping/physionet.org/files/ludb/1.0.1/data',
    )
    DATA_ROOT = first_existing(
        Path('/Users/vishwam/VSCode/physionet.org/files'),
        Path('/Users/vishwam/VSCode/PQRST_mapping/physionet.org/files'),
        Path.cwd() / 'physionet.org/files',
        Path.cwd() / 'PQRST_mapping/physionet.org/files',
    )
    DRIVE_ROOT = Path.cwd()
    ARTIFACT_DIR = Path.cwd() / 'reviewer2_artifacts'

CHECKPOINT_DIR = ARTIFACT_DIR / 'checkpoints'
STATE_DIR = ARTIFACT_DIR / 'state'
for folder in [DATA_ROOT, ARTIFACT_DIR, CHECKPOINT_DIR, STATE_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

PROFILE = 'matched_ablation'
PROFILES = {
    'matched_ablation': {'seeds': [1, 2, 3], 'epochs': 15},
}
if PROFILE not in PROFILES: raise ValueError(PROFILE)
TRAINING_SEEDS = PROFILES[PROFILE]['seeds']
NUM_EPOCHS = PROFILES[PROFILE]['epochs']
PRE, POST, R5_POST = 120, 240, 320
BATCH_SIZE = 64
R6_ADAPT_EPOCHS = 8
R6_ADAPT_FRACTION = 0.10
QTDB_SPLIT_SEED = 7
R6_SPLIT_SEED = 17

if hasattr(torch, 'mps') and torch.mps.is_available():
    DEVICE = torch.device('mps')
elif torch.cuda.is_available():
    DEVICE = torch.device('cuda')
else:
    DEVICE = torch.device('cpu')

print('PROFILE:', PROFILE, '| seeds:', TRAINING_SEEDS, '| epochs:', NUM_EPOCHS, '| device:', DEVICE)
print('Reviewer protocol: 3 independent seeds with 15 epochs per seed due to computational constraints.')
print('Selected device priority: MPS -> CUDA -> CPU')
print('QTDB_DIR:', QTDB_DIR)
print('LUDB_DIR:', LUDB_DIR)
print('ARTIFACT_DIR:', ARTIFACT_DIR)


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
PROFILE: matched_ablation | seeds: [1, 2, 3] | epochs: 15 | device: mps
Reviewer protocol: 3 independent seeds with 15 epochs per seed due to computational constraints.
Selected device priority: MPS -> CUDA -> CPU
QTDB_DIR: /Users/vishwam/VSCode/physionet.org/files/qtdb/1.0.0
LUDB_DIR: /Users/vishwam/VSCode/PQRST_mapping/physionet.org/files/ludb/1.0.1/data
ARTIFACT_DIR: /Users/vishwam/VSCode/PQRST_mapping/reviewer2_artifacts


## Dataset sources and persistence

Official sources:
- QTDB: https://physionet.org/content/qtdb/1.0.0/
- LUDB: https://physionet.org/content/ludb/1.0.1/

The download cell uses WFDB and writes directly to Drive. It skips a dataset when paired `.hea` and signal files already exist, so a new Colab session does not redownload data.

In [30]:
import wfdb

def paired_records(folder):
    headers = {path.stem for path in Path(folder).glob('*.hea')}
    signals = {path.stem for path in Path(folder).glob('*.dat')}
    return sorted(headers & signals)

def download_once(database, target, expected_count):
    target = Path(target); target.mkdir(parents=True, exist_ok=True)
    complete = target / '.download_complete.json'
    records = paired_records(target)
    if complete.exists() and len(records) == expected_count:
        print(database, 'already present:', len(records), 'records')
        return records
    print('Downloading', database, 'to', target, '- this is persisted on Drive')
    wfdb.dl_database(database, dl_dir=str(target), keep_subdirs=False)
    records = paired_records(target)
    if len(records) != expected_count:
        raise RuntimeError(f'{database}: expected {expected_count} paired records, found {len(records)}')
    complete.write_text(json.dumps({'database': database, 'records': records, 'time': time.time()}, indent=2))
    return records

qtdb_records = download_once('qtdb', QTDB_DIR, 105)
ludb_records = download_once('ludb', LUDB_DIR, 200)
print('QTDB:', len(qtdb_records), '| LUDB:', len(ludb_records))

qtdb already present: 105 records
ludb already present: 200 records
QTDB: 105 | LUDB: 200


## Persistence and crash recovery

Every write goes through a temporary file and atomic rename. The epoch checkpoint contains model weights, optimizer state, best validation state, current epoch, seed, profile, and provenance. If Colab disconnects, remount Drive, rerun setup/import cells, and rerun the training cell; completed epochs and seeds are skipped or resumed.

In [31]:
import tempfile
import torch

def atomic_torch_save(value, path):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    with tempfile.NamedTemporaryFile(dir=path.parent, suffix='.tmp', delete=False) as handle:
        temporary = Path(handle.name)
    try:
        torch.save(value, temporary)
        temporary.replace(path)
    finally:
        temporary.unlink(missing_ok=True)

def atomic_json_save(value, path):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + '.tmp')
    temporary.write_text(json.dumps(value, indent=2, default=lambda item: item.item() if isinstance(item, np.generic) else str(item)))
    temporary.replace(path)

def checkpoint_path(seed, name, epoch=None):
    suffix = 'latest' if epoch is None else f'epoch_{epoch:02d}'
    return CHECKPOINT_DIR / f'{PROFILE}_seed_{seed}_{name}_{suffix}.pth'

def save_npz_atomic(path, **arrays):
    path = Path(path); temporary = path.with_suffix('.tmp.npz')
    np.savez(temporary, **arrays)
    temporary.replace(path)

def save_seed_progress(seed, payload):
    atomic_json_save(payload, STATE_DIR / f'{PROFILE}_seed_{seed}_progress.json')

print('Persistent artifact root:', ARTIFACT_DIR)

Persistent artifact root: /Users/vishwam/VSCode/PQRST_mapping/reviewer2_artifacts


In [32]:
# Record-level splits and train-only normalization. This cell is intentionally resumable.
from sklearn.model_selection import train_test_split
from scipy.signal import butter, sosfiltfilt, find_peaks, resample_poly

REVIEWER_NOTEBOOK = first_existing(
    Path.cwd() / 'reviewer2_matched_metrics.ipynb',
    Path.cwd() / 'PQRST_mapping/reviewer2_matched_metrics.ipynb',
    DRIVE_ROOT / 'reviewer2_matched_metrics.ipynb',
)
if not REVIEWER_NOTEBOOK.exists():
    raise FileNotFoundError(
        'Local reviewer2_matched_metrics.ipynb was not found. Place it beside this notebook or in the project root.'
    )
reviewer_nb = json.loads(REVIEWER_NOTEBOOK.read_text())

# Load only shared helper definitions from the local reviewer notebook; do not execute stale outputs or training.
helper_source = ''.join(reviewer_nb['cells'][3]['source'])
exec(helper_source, globals())

qtdb_train_records, qtdb_val_records = train_test_split(
    qtdb_records,
    test_size=0.20,
    random_state=QTDB_SPLIT_SEED,
    shuffle=True,
)
qtdb_train_records, qtdb_val_records = sorted(qtdb_train_records), sorted(qtdb_val_records)
r6_adapt_records, r6_test_records = train_test_split(
    ludb_records,
    test_size=1.0 - R6_ADAPT_FRACTION,
    random_state=R6_SPLIT_SEED,
    shuffle=True,
)
r6_adapt_records, r6_test_records = sorted(r6_adapt_records), sorted(r6_test_records)
assert len(qtdb_train_records) == 84 and len(qtdb_val_records) == 21
assert len(r6_adapt_records) == 20 and len(r6_test_records) == 180
assert set(qtdb_train_records).isdisjoint(qtdb_val_records)
assert set(r6_adapt_records).isdisjoint(r6_test_records)

# Execute the local reviewed partition builder so windows are created only after record splits.
partition_source = ''.join(reviewer_nb['cells'][5]['source'])
exec(partition_source, globals())
manifest = {
    'profile': PROFILE,
    'training_seeds': TRAINING_SEEDS,
    'epochs': NUM_EPOCHS,
    'qtdb_directory': str(QTDB_DIR),
    'ludb_directory': str(LUDB_DIR),
    'qtdb_train_records': qtdb_train_records,
    'qtdb_validation_records': qtdb_val_records,
    'ludb_adaptation_records': r6_adapt_records,
    'ludb_test_records': r6_test_records,
    'split_seeds': {'qtdb': QTDB_SPLIT_SEED, 'ludb': R6_SPLIT_SEED},
    'r6_adaptation_fraction': R6_ADAPT_FRACTION,
    'r6_adaptation_epochs': R6_ADAPT_EPOCHS,
}
manifest.update({
    'qtdb_train_windows_240': len(X_qt_train),
    'qtdb_validation_windows_240': len(X_qt_val),
    'ludb_adaptation_windows_240': len(X_lu_adapt_240),
    'ludb_test_windows_240': len(X_lu_test_240),
    'qtdb_train_mean_240': train_mean,
    'qtdb_train_std_240': train_std,
})
atomic_json_save(manifest, ARTIFACT_DIR / 'colab_run_manifest.json')
np.savez(ARTIFACT_DIR / 'reviewer2_qtdb_normalization.npz', mean=train_mean, std=train_std)
print('Preprocessing and manifest persisted:', ARTIFACT_DIR / 'colab_run_manifest.json')

QTDB total records: 105 | train: 84 | validation: 21
QTDB train records: ['sel100', 'sel102', 'sel103', 'sel104', 'sel114', 'sel117', 'sel123', 'sel14046', 'sel14157', 'sel14172', 'sel15814', 'sel16273', 'sel16420', 'sel16483', 'sel16539', 'sel16773', 'sel16786', 'sel17152', 'sel213', 'sel221', 'sel223', 'sel231', 'sel232', 'sel233', 'sel30', 'sel301', 'sel302', 'sel306', 'sel307', 'sel308', 'sel31', 'sel310', 'sel32', 'sel33', 'sel34', 'sel35', 'sel36', 'sel37', 'sel38', 'sel39', 'sel41', 'sel42', 'sel45', 'sel46', 'sel47', 'sel48', 'sel49', 'sel50', 'sel51', 'sel52', 'sel808', 'sel811', 'sel820', 'sel821', 'sel840', 'sel847', 'sel853', 'sel871', 'sel872', 'sel873', 'sel883', 'sele0104', 'sele0106', 'sele0110', 'sele0114', 'sele0116', 'sele0121', 'sele0124', 'sele0126', 'sele0129', 'sele0133', 'sele0136', 'sele0166', 'sele0170', 'sele0203', 'sele0210', 'sele0303', 'sele0405', 'sele0409', 'sele0603', 'sele0604', 'sele0606', 'sele0607', 'sele0612']
QTDB validation records: ['sel116', 's

## Resumable training harness

This is the control layer to place around the existing reviewer model/preprocessing functions. It checkpoints after every epoch, writes a progress manifest after every epoch, and writes a completed-seed result immediately. For a full run, use the model/training function definitions from the reviewer notebook in the next cell or paste them into this persistent notebook.

The guard below refuses to start the expensive loop until the required functions are defined, avoiding an accidental partial run.

In [26]:
# Load reviewed model/training definitions automatically from the persisted source notebook.
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import classification_report, f1_score, accuracy_score, precision_score, recall_score

model_source = next(''.join(cell['source']) for cell in reviewer_nb['cells'] if any('class CNNFeatureExtractor' in line for line in cell['source']))
exec(model_source.split("# R4 threshold is selected", 1)[0], globals())
training_source = next(''.join(cell['source']) for cell in reviewer_nb['cells'] if any('def train_fixed_model' in line for line in cell['source']))
exec(training_source.split('seed_rows, seed_provenance', 1)[0], globals())

REQUIRED_FUNCTIONS = ['RPeakGuidedML2', 'RPeakTimeML2', 'FocalLoss', 'make_loader', 'run_training_epoch', 'predict', 'time_channel']
missing = [name for name in REQUIRED_FUNCTIONS if name not in globals()]
if missing: raise RuntimeError(f'Reviewer definitions failed to load: {missing}')


def resumable_train(model, train_features, train_labels, val_features, val_labels, criterion, seed, name):
    set_global_seed(seed)
    train_loader = make_loader(train_features, train_labels, seed, shuffle=True)
    val_loader = make_loader(val_features, val_labels, seed, shuffle=False)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    latest_path = checkpoint_path(seed, name)
    start_epoch, best_loss, best_epoch, best_state = 0, float('inf'), 0, None
    if latest_path.exists():
        state = torch.load(latest_path, map_location=DEVICE)
        model.load_state_dict(state['model_state_dict'])
        optimizer.load_state_dict(state['optimizer_state_dict'])
        start_epoch = int(state.get('epoch', 0))
        best_loss = float(state.get('best_validation_loss', float('inf')))
        best_epoch = int(state.get('best_epoch', 0))
        best_state = state.get('best_model_state_dict')
        print(f'Resuming {name}, seed {seed}, after epoch {start_epoch}')
    for epoch in range(start_epoch, NUM_EPOCHS):
        train_loss = run_training_epoch(model, train_loader, criterion, optimizer)
        val_loss = run_training_epoch(model, val_loader, criterion)
        if val_loss < best_loss:
            best_loss, best_epoch = val_loss, epoch + 1
            best_state = {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}
        payload = {'epoch': epoch + 1, 'best_validation_loss': best_loss, 'best_epoch': best_epoch, 'seed': seed, 'name': name, 'profile': PROFILE, 'model_state_dict': model.state_dict(), 'optimizer_state_dict': optimizer.state_dict(), 'best_model_state_dict': best_state}
        atomic_torch_save(payload, latest_path)
        atomic_torch_save(payload, checkpoint_path(seed, name, epoch + 1))
        save_seed_progress(seed, {'last_completed_model': name, 'last_completed_epoch': epoch + 1, 'best_epoch': best_epoch, 'best_validation_loss': best_loss, 'profile': PROFILE, 'time': time.time()})
        print(f'{name} seed={seed} epoch={epoch + 1}/{NUM_EPOCHS}: train={train_loss:.4f} val={val_loss:.4f} persisted')
    if best_state is not None: model.load_state_dict(best_state)
    return model, {'best_epoch': best_epoch, 'best_validation_loss': best_loss}


def completed_seed_path(seed):
    return STATE_DIR / f'{PROFILE}_seed_{seed}_complete.json'


def save_completed_seed(seed, rows, provenance):
    atomic_json_save({'seed': seed, 'metrics': rows, 'provenance': provenance, 'completed_at': time.time()}, completed_seed_path(seed))
    pd.DataFrame(rows).to_csv(ARTIFACT_DIR / f'{PROFILE}_seed_{seed}_metrics.csv', index=False)

print('Definitions loaded; resumable training is ready.')

Definitions loaded; resumable training is ready.


In [27]:
seed_rows = []


def run_seed_experiment(seed):
    global seed_rows
    completed = completed_seed_path(seed)
    if completed.exists():
        saved = json.loads(completed.read_text())
        seed_rows.extend(saved['metrics'])
        print('Seed already complete; loaded persisted results:', seed)
        return pd.DataFrame(saved['metrics'])

    seed_provenance = {
        'seed': seed,
        'profile': PROFILE,
        'qtdb_train_records': qtdb_train_records,
        'qtdb_validation_records': qtdb_val_records,
        'ludb_adaptation_records': r6_adapt_records,
        'ludb_test_records': r6_test_records,
        'qtdb_train_mean_240': train_mean,
        'qtdb_train_std_240': train_std,
        'r6_adaptation_epochs': R6_ADAPT_EPOCHS,
    }
    rows = []

    r1, r1_info = resumable_train(RPeakGuidedML2().to(DEVICE), X_qt_train[:, None], Y_qt_train, X_qt_val[:, None], Y_qt_val, nn.CrossEntropyLoss(), seed, 'R1')
    r1_info['post_samples'] = POST
    rows.append(classification_row('R1', Y_lu_test_240, predict(r1, X_lu_test_240[:, None]), lu_test_ids_240, POST) | {'Seed': seed, **r1_info})

    class_counts = np.bincount(Y_qt_train.ravel(), minlength=3).astype(np.float32)
    weights = class_counts.sum() / np.maximum(class_counts, 1.0)
    weights = torch.tensor(weights / weights.mean(), dtype=torch.float32, device=DEVICE)
    r2, r2_info = resumable_train(RPeakGuidedML2().to(DEVICE), X_qt_train[:, None], Y_qt_train, X_qt_val[:, None], Y_qt_val, FocalLoss(alpha=weights), seed, 'R2')
    r2_info['post_samples'] = POST
    rows.append(classification_row('R2', Y_lu_test_240, predict(r2, X_lu_test_240[:, None]), lu_test_ids_240, POST) | {'Seed': seed, **r2_info})

    r3, r3_info = resumable_train(RPeakGuidedML2().to(DEVICE), X_qt_train[:, None], Y_qt_train, X_qt_val[:, None], Y_qt_val, FocalLoss(), seed, 'R3')
    r3_info['post_samples'] = POST
    rows.append(classification_row('R3', Y_lu_test_240, predict(r3, X_lu_test_240[:, None]), lu_test_ids_240, POST) | {'Seed': seed, **r3_info})

    r4_val_probabilities = predict_probabilities(r1, X_qt_val[:, None])
    threshold_grid = []
    for threshold in np.arange(0.10, 0.76, 0.05):
        decoded = r4_decode(r4_val_probabilities, threshold)
        threshold_grid.append((threshold, f1_score(Y_qt_val.ravel(), decoded.ravel(), labels=[0, 1, 2], average='macro', zero_division=0)))
    r4_threshold = float(max(threshold_grid, key=lambda item: item[1])[0])
    r4_prediction = r4_decode(predict_probabilities(r1, X_lu_test_240[:, None]), r4_threshold)
    rows.append(classification_row('R4', Y_lu_test_240, r4_prediction, lu_test_ids_240, POST) | {'Seed': seed, 'Best epoch': r1_info['best_epoch'], 'Best validation loss': r1_info['best_validation_loss'], 'R4 threshold': r4_threshold})

    r5, r5_info = resumable_train(RPeakGuidedML2().to(DEVICE), X_qt_train_320[:, None], Y_qt_train_320, X_qt_val_320[:, None], Y_qt_val_320, nn.CrossEntropyLoss(), seed, 'R5')
    r5_info['post_samples'] = R5_POST
    rows.append(classification_row('R5', Y_lu_test_320, predict(r5, X_lu_test_320[:, None]), lu_test_ids_320, R5_POST) | {'Seed': seed, **r5_info})

    r6, r6_info = resumable_train(RPeakTimeML2().to(DEVICE), time_channel(X_qt_train_320, R5_POST), Y_qt_train_320, time_channel(X_qt_val_320, R5_POST), Y_qt_val_320, nn.CrossEntropyLoss(), seed, 'R6_QTDB')
    adaptation_loader = make_loader(time_channel(X_lu_adapt_320, R5_POST), Y_lu_adapt_320, seed, shuffle=True)
    adaptation_optimizer = torch.optim.Adam(r6.parameters(), lr=1e-4)
    adaptation_state_path = checkpoint_path(seed, 'R6_adaptation')
    adaptation_start = 0
    if adaptation_state_path.exists():
        adaptation_state = torch.load(adaptation_state_path, map_location=DEVICE)
        r6.load_state_dict(adaptation_state['model_state_dict'])
        adaptation_optimizer.load_state_dict(adaptation_state['optimizer_state_dict'])
        adaptation_start = int(adaptation_state.get('epoch', 0))
    for adaptation_epoch in range(adaptation_start, R6_ADAPT_EPOCHS):
        adaptation_loss = run_training_epoch(r6, adaptation_loader, nn.CrossEntropyLoss(), adaptation_optimizer)
        atomic_torch_save({'epoch': adaptation_epoch + 1, 'seed': seed, 'name': 'R6_adaptation', 'model_state_dict': r6.state_dict(), 'optimizer_state_dict': adaptation_optimizer.state_dict(), 'adaptation_loss': adaptation_loss}, adaptation_state_path)
        print(f'R6 adaptation seed={seed} epoch={adaptation_epoch + 1}/{R6_ADAPT_EPOCHS}: loss={adaptation_loss:.4f} persisted')
    rows.append(classification_row('R6 adapted', Y_lu_test_320, predict(r6, time_channel(X_lu_test_320, R5_POST)), lu_test_ids_320, R5_POST) | {'Seed': seed, 'Best epoch': r6_info['best_epoch'], 'Best validation loss': r6_info['best_validation_loss'], 'R6 adaptation epochs': R6_ADAPT_EPOCHS})

    seed_rows.extend(rows)
    save_completed_seed(seed, rows, seed_provenance)
    pd.DataFrame(seed_rows).to_csv(ARTIFACT_DIR / f'{PROFILE}_metrics_so_far.csv', index=False)
    print('Completed and persisted seed:', seed)
    return pd.DataFrame(rows)

print('Seed runner ready. Run one seed cell at a time.')


Seed runner ready. Run one seed cell at a time.


In [14]:
# Seed 1
run_seed_experiment(1)


Resuming R1, seed 1, after epoch 15
R2 seed=1 epoch=1/15: train=0.0946 val=0.1554 persisted
R2 seed=1 epoch=2/15: train=0.0949 val=0.0571 persisted
R2 seed=1 epoch=3/15: train=0.0507 val=0.0508 persisted
R2 seed=1 epoch=4/15: train=0.0427 val=0.0504 persisted
R2 seed=1 epoch=5/15: train=0.0392 val=0.0497 persisted
R2 seed=1 epoch=6/15: train=0.0356 val=0.0428 persisted
R2 seed=1 epoch=7/15: train=0.0344 val=0.0455 persisted
R2 seed=1 epoch=8/15: train=0.0319 val=0.0460 persisted
R2 seed=1 epoch=9/15: train=0.0307 val=0.0466 persisted
R2 seed=1 epoch=10/15: train=0.0291 val=0.0443 persisted
R2 seed=1 epoch=11/15: train=0.0288 val=0.0455 persisted
R2 seed=1 epoch=12/15: train=0.0291 val=0.0433 persisted
R2 seed=1 epoch=13/15: train=0.0267 val=0.0452 persisted
R2 seed=1 epoch=14/15: train=0.0263 val=0.0488 persisted
R2 seed=1 epoch=15/15: train=0.0260 val=0.0436 persisted
R3 seed=1 epoch=1/15: train=0.1433 val=0.0827 persisted
R3 seed=1 epoch=2/15: train=0.1267 val=0.0955 persisted
R3 see

,Experiment,Records,Windows,Post samples,Accuracy,P precision,P recall,P F1,T precision,T recall,...,Macro F1,Weighted F1,Seed,best_epoch,best_validation_loss,post_samples,Best epoch,Best validation loss,R4 threshold,R6 adaptation epochs
0,R1,180,1710,240,0.881831,0.778594,0.800453,0.789372,0.768349,0.727259,...,0.819305,0.881368,1,14.0,0.194496,240.0,NaN,NaN,NaN,NaN
1,R2,180,1710,240,0.855689,0.688453,0.869567,0.768483,0.655985,0.835386,...,0.800885,0.860865,1,6.0,0.042836,240.0,NaN,NaN,NaN,NaN
2,R3,180,1710,240,0.874241,0.752113,0.768278,0.760109,0.756783,0.735622,...,0.807444,0.874087,1,11.0,0.056653,240.0,NaN,NaN,NaN,NaN
3,R4,180,1710,240,0.839750,0.775884,0.424571,0.548822,0.799012,0.574435,...,0.705313,0.826168,1,NaN,NaN,NaN,14.0,0.194496,0.55,NaN
4,R5,180,1655,320,0.878075,0.771089,0.831141,0.799990,0.772157,0.758271,...,0.827170,0.878311,1,13.0,0.190187,320.0,NaN,NaN,NaN,NaN
5,R6 adapted,180,1655,320,0.889386,0.786752,0.837123,0.811156,0.781606,0.817578,...,0.844506,0.890248,1,NaN,NaN,NaN,14.0,0.186295,NaN,8.0


In [15]:
# Seed 2
run_seed_experiment(2)


R1 seed=2 epoch=1/15: train=0.3570 val=0.2492 persisted
R1 seed=2 epoch=2/15: train=0.2108 val=0.2200 persisted
R1 seed=2 epoch=3/15: train=0.1765 val=0.2073 persisted
R1 seed=2 epoch=4/15: train=0.1650 val=0.2025 persisted
R1 seed=2 epoch=5/15: train=0.1574 val=0.2260 persisted
R1 seed=2 epoch=6/15: train=0.1467 val=0.2316 persisted
R1 seed=2 epoch=7/15: train=0.1405 val=0.2100 persisted
R1 seed=2 epoch=8/15: train=0.1359 val=0.2048 persisted
R1 seed=2 epoch=9/15: train=0.1314 val=0.2121 persisted
R1 seed=2 epoch=10/15: train=0.1296 val=0.1918 persisted
R1 seed=2 epoch=11/15: train=0.1257 val=0.2093 persisted
R1 seed=2 epoch=12/15: train=0.1232 val=0.2043 persisted
R1 seed=2 epoch=13/15: train=0.1224 val=0.1928 persisted
R1 seed=2 epoch=14/15: train=0.1193 val=0.1983 persisted
R1 seed=2 epoch=15/15: train=0.1176 val=0.2079 persisted
R2 seed=2 epoch=1/15: train=0.0969 val=0.0587 persisted
R2 seed=2 epoch=2/15: train=0.0598 val=0.0531 persisted
R2 seed=2 epoch=3/15: train=0.0450 val=0.0

,Experiment,Records,Windows,Post samples,Accuracy,P precision,P recall,P F1,T precision,T recall,...,Macro F1,Weighted F1,Seed,best_epoch,best_validation_loss,post_samples,Best epoch,Best validation loss,R4 threshold,R6 adaptation epochs
0,R1,180,1710,240,0.877714,0.772064,0.801363,0.786440,0.738862,0.750645,...,0.816372,0.878194,2,10.0,0.191788,240.0,NaN,NaN,NaN,NaN
1,R2,180,1710,240,0.855270,0.697929,0.904889,0.788048,0.647067,0.797608,...,0.800652,0.860228,2,11.0,0.039903,240.0,NaN,NaN,NaN,NaN
2,R3,180,1710,240,0.881685,0.763834,0.841246,0.800673,0.747392,0.756607,...,0.824288,0.882516,2,15.0,0.058277,240.0,NaN,NaN,NaN,NaN
3,R4,180,1710,240,0.841061,0.774718,0.431615,0.554374,0.793443,0.588860,...,0.709902,0.828315,2,NaN,NaN,NaN,10.0,0.191788,0.55,NaN
4,R5,180,1655,320,0.876262,0.772890,0.824868,0.798033,0.793354,0.711736,...,0.821416,0.875210,2,12.0,0.179164,320.0,NaN,NaN,NaN,NaN
5,R6 adapted,180,1655,320,0.889816,0.792157,0.832967,0.812050,0.779627,0.820288,...,0.845020,0.890670,2,NaN,NaN,NaN,11.0,0.178472,NaN,8.0


In [16]:
# Seed 3
run_seed_experiment(3)


R1 seed=3 epoch=1/15: train=0.3295 val=0.2438 persisted
R1 seed=3 epoch=2/15: train=0.2035 val=0.1970 persisted
R1 seed=3 epoch=3/15: train=0.1681 val=0.2409 persisted
R1 seed=3 epoch=4/15: train=0.2353 val=0.2471 persisted
R1 seed=3 epoch=5/15: train=0.1821 val=0.2161 persisted
R1 seed=3 epoch=6/15: train=0.1539 val=0.3530 persisted
R1 seed=3 epoch=7/15: train=0.1532 val=0.1989 persisted
R1 seed=3 epoch=8/15: train=0.1409 val=0.2011 persisted
R1 seed=3 epoch=9/15: train=0.1361 val=0.2078 persisted
R1 seed=3 epoch=10/15: train=0.1322 val=0.1990 persisted
R1 seed=3 epoch=11/15: train=0.1293 val=0.1865 persisted
R1 seed=3 epoch=12/15: train=0.1262 val=0.1934 persisted
R1 seed=3 epoch=13/15: train=0.1235 val=0.1978 persisted
R1 seed=3 epoch=14/15: train=0.1215 val=0.2015 persisted
R1 seed=3 epoch=15/15: train=0.1191 val=0.2911 persisted
R2 seed=3 epoch=1/15: train=0.1132 val=0.0764 persisted
R2 seed=3 epoch=2/15: train=0.0706 val=0.0562 persisted
R2 seed=3 epoch=3/15: train=0.0461 val=0.0

,Experiment,Records,Windows,Post samples,Accuracy,P precision,P recall,P F1,T precision,T recall,...,Macro F1,Weighted F1,Seed,best_epoch,best_validation_loss,post_samples,Best epoch,Best validation loss,R4 threshold,R6 adaptation epochs
0,R1,180,1710,240,0.880458,0.756835,0.813906,0.784334,0.760949,0.745495,...,0.819207,0.880790,3,11.0,0.186529,240.0,NaN,NaN,NaN,NaN
1,R2,180,1710,240,0.863908,0.706380,0.890079,0.787660,0.666092,0.844414,...,0.812473,0.868719,3,13.0,0.038999,240.0,NaN,NaN,NaN,NaN
2,R3,180,1710,240,0.885203,0.777409,0.784142,0.780761,0.792002,0.737436,...,0.822699,0.884479,3,6.0,0.056775,240.0,NaN,NaN,NaN,NaN
3,R4,180,1710,240,0.840689,0.757863,0.430605,0.549177,0.800215,0.587880,...,0.708749,0.827948,3,NaN,NaN,NaN,11.0,0.186529,0.55,NaN
4,R5,180,1655,320,0.878859,0.771524,0.819919,0.794986,0.778204,0.761036,...,0.827159,0.878976,3,15.0,0.198142,320.0,NaN,NaN,NaN,NaN
5,R6 adapted,180,1655,320,0.888991,0.785947,0.834092,0.809304,0.775935,0.827945,...,0.844372,0.890051,3,NaN,NaN,NaN,8.0,0.175323,NaN,8.0


In [33]:
from scipy.stats import t as student_t


def mean_sd_ci(values, confidence=0.95):
    values = np.asarray(values, dtype=float)
    mean = float(values.mean())
    if len(values) < 2:
        return mean, 0.0, np.nan, np.nan
    sd = float(values.std(ddof=1))
    critical = float(student_t.ppf((1.0 + confidence) / 2.0, len(values) - 1))
    margin = critical * sd / np.sqrt(len(values))
    return mean, sd, mean - margin, mean + margin


def load_completed_seed_metrics():
    persisted_rows = []
    for path in sorted(STATE_DIR.glob(f'{PROFILE}_seed_*_complete.json')):
        saved = json.loads(path.read_text())
        persisted_rows.extend(saved.get('metrics', []))
    if persisted_rows:
        return pd.DataFrame(persisted_rows)
    if seed_rows:
        return pd.DataFrame(seed_rows)
    raise RuntimeError('No completed seed metrics found. Run the seed cells before generating summaries.')


def summarize_seed_metrics(df):
    summary_rows = []
    for experiment, group in df.groupby('Experiment'):
        for metric in ['P F1', 'T F1', 'Macro F1', 'Weighted F1', 'Accuracy']:
            values = np.asarray(group[metric], dtype=float)
            mean, sd, low, high = mean_sd_ci(values)
            summary_rows.append({'Experiment': experiment, 'Metric': metric, 'Seeds': len(group), 'Mean': mean, 'SD': sd, '95% CI low': low, '95% CI high': high})
    return pd.DataFrame(summary_rows)


seed_metrics = load_completed_seed_metrics()
summary = summarize_seed_metrics(seed_metrics)
summary.to_csv(ARTIFACT_DIR / f'{PROFILE}_mean_sd_t_ci.csv', index=False)

pivot = seed_metrics.pivot(index='Seed', columns='Experiment', values='Macro F1')
if {'R3', 'R6 adapted'} <= set(pivot.columns):
    differences = pivot['R6 adapted'] - pivot['R3']
    mean, sd, low, high = mean_sd_ci(differences)
    difference_table = pd.DataFrame({'Seed': differences.index, 'R3 Macro F1': pivot['R3'], 'R6 Macro F1': pivot['R6 adapted'], 'R6 minus R3': differences})
    difference_summary = pd.DataFrame([{'Comparison': 'R6 adapted - R3', 'Mean': mean, 'SD': sd, '95% CI low': low, '95% CI high': high}])
    difference_table.to_csv(ARTIFACT_DIR / f'{PROFILE}_r3_r6_by_seed.csv', index=False)
    difference_summary.to_csv(ARTIFACT_DIR / f'{PROFILE}_r3_r6_difference_ci.csv', index=False)
    display(difference_table.round(4)); display(difference_summary.round(4))

summary_rows = summary.to_dict('records')
display(summary.round(4))
print('All summaries persisted to:', ARTIFACT_DIR)

,Seed,R3 Macro F1,R6 Macro F1,R6 minus R3
Seed,,,,
1,1,0.8074,0.8445,0.0371
2,2,0.8243,0.8450,0.0207
3,3,0.8227,0.8444,0.0217


,Comparison,Mean,SD,95% CI low,95% CI high
0,R6 adapted - R3,0.0265,0.0092,0.0037,0.0493


,Experiment,Metric,Seeds,Mean,SD,95% CI low,95% CI high
0,R1,P F1,3,0.7867,0.0025,0.7804,0.7930
1,R1,T F1,3,0.7484,0.0043,0.7376,0.7591
2,R1,Macro F1,3,0.8183,0.0017,0.8142,0.8224
3,R1,Weighted F1,3,0.8801,0.0017,0.8759,0.8843
4,R1,Accuracy,3,0.8800,0.0021,0.8748,0.8852
5,R2,P F1,3,0.7814,0.0112,0.7536,0.8092
6,R2,T F1,3,0.7314,0.0154,0.6931,0.7697
7,R2,Macro F1,3,0.8047,0.0068,0.7879,0.8215
8,R2,Weighted F1,3,0.8633,0.0047,0.8515,0.8750
9,R2,Accuracy,3,0.8583,0.0049,0.8462,0.8704


All summaries persisted to: /Users/vishwam/VSCode/PQRST_mapping/reviewer2_artifacts


In [34]:
from datetime import datetime, timezone

reviewer2_checks = {
    'QTDB complete population': len(qtdb_records) == 105,
    'QTDB split 84/21': len(qtdb_train_records) == 84 and len(qtdb_val_records) == 21,
    'LUDB complete population': len(ludb_records) == 200,
    'LUDB split 20/180': len(r6_adapt_records) == 20 and len(r6_test_records) == 180,
    'QTDB record disjointness': set(qtdb_train_records).isdisjoint(qtdb_val_records),
    'LUDB record disjointness': set(r6_adapt_records).isdisjoint(r6_test_records),
    'train-only normalization persisted': (ARTIFACT_DIR / 'reviewer2_qtdb_normalization.npz').exists(),
    'split manifest persisted': (ARTIFACT_DIR / 'colab_run_manifest.json').exists(),
    'three seeds configured': TRAINING_SEEDS == [1, 2, 3],
    'matched-ablation 15-epoch protocol selected': PROFILE == 'matched_ablation' and NUM_EPOCHS == 15,
}
completed_seeds = sorted(int(path.stem.split('_seed_')[1].split('_')[0]) for path in STATE_DIR.glob(f'{PROFILE}_seed_*_complete.json'))
reviewer2_checks['three seeds completed'] = completed_seeds == [1, 2, 3]
final_reviewer2_results = {
    'generated_at_utc': datetime.now(timezone.utc).isoformat(),
    'status': 'PASS' if all(reviewer2_checks.values()) else 'INCOMPLETE',
    'checks': reviewer2_checks,
    'profile': PROFILE,
    'completed_seeds': completed_seeds,
    'dataset_counts': {'qtdb_total': len(qtdb_records), 'qtdb_train': len(qtdb_train_records), 'qtdb_validation': len(qtdb_val_records), 'ludb_total': len(ludb_records), 'ludb_adaptation': len(r6_adapt_records), 'ludb_test': len(r6_test_records)},
    'artifacts': {'manifest': str(ARTIFACT_DIR / 'colab_run_manifest.json'), 'normalization': str(ARTIFACT_DIR / 'reviewer2_qtdb_normalization.npz'), 'checkpoint_directory': str(CHECKPOINT_DIR), 'state_directory': str(STATE_DIR), 'metrics': str(ARTIFACT_DIR / f'{PROFILE}_metrics.csv')},
    'limitations': ['Quick and pilot profiles are validation runs, not Reviewer 2 final results.', 'The current target representation is background/P/T; QRS delineation is not claimed.', 'No event-boundary or external-delineator result is included in this Reviewer 2 package.'],
}
atomic_json_save(final_reviewer2_results, ARTIFACT_DIR / 'FINAL_REVIEWER_2_RESULTS.json')
for name, passed in reviewer2_checks.items(): print(f'{"PASS" if passed else "FAIL"}: {name}')
print('FINAL_REVIEWER_2_RESULTS:', final_reviewer2_results['status'])

PASS: QTDB complete population
PASS: QTDB split 84/21
PASS: LUDB complete population
PASS: LUDB split 20/180
PASS: QTDB record disjointness
PASS: LUDB record disjointness
PASS: train-only normalization persisted
PASS: split manifest persisted
PASS: three seeds configured
PASS: matched-ablation 15-epoch protocol selected
PASS: three seeds completed
FINAL_REVIEWER_2_RESULTS: PASS


## Recommended cloud runtimes

- Main reporting protocol: Colab L4 or T4 with the fixed `matched_ablation` configuration.
- For a cheap smoke check, run a single seed locally or in Colab before the three-seed run.
- Avoid CPU runtimes for the main matched-ablation report.
- A100 is optional but not required for the final reviewer-targeted protocol.

Practical workflow:
1. Run setup, download, persistence, and split cells once.
2. Run one smoke seed only to confirm checkpointing and data integrity.
3. Execute the three matched seeds [1, 2, 3] at 15 epochs each.
4. After each seed, verify `reviewer2_artifacts/state/` and `checkpoints/` in Drive.
5. If Colab crashes, reconnect, remount Drive, rerun setup/import cells, and resume from the latest checkpoint.

Drive persistence protects files, but it cannot recover a training step that was never checkpointed. Keep checkpoint frequency at every epoch.